# CDK20_HUMAN: 相同蛋白の探索と既知活性化合物の収集

CDK20_HUMANはまだX線結晶構造が解かれていない。そこで、

1. UniProt/BLASTでCDK20に配列が近い蛋白を探す
2. 見つかった蛋白ごとにPDBエントリ数・ChEMBL活性化合物数を集計し、この研究で参照蛋白として活用できそうなものをランキングする
3. CDK20自体の既知の活性化合物をChEMBLから収集する


In [1]:
from pathlib import Path

TARGET = "CDK20_HUMAN"
OUTDIR = Path("data/cdk20_investigation")
OUTDIR.mkdir(parents=True, exist_ok=True)

In [2]:
from idmap import resolve_uniprot_accession

accession = resolve_uniprot_accession(TARGET)
print(accession)

02:34:22 [idmap.identifiers] Resolving UniProt entry name CDK20_HUMAN -> accession ...
02:34:23 [idmap.identifiers]   -> Q8IZL9


Q8IZL9


In [3]:
from uniprot.entry import fetch_fasta

fasta = fetch_fasta(accession).decode()
sequence = "".join(line for line in fasta.splitlines() if not line.startswith(">"))
print(fasta)
print(f"sequence length: {len(sequence)}")

02:34:23 [uniprot.entry] Fetching UniProt FASTA for Q8IZL9 ...


>sp|Q8IZL9|CDK20_HUMAN Cyclin-dependent kinase 20 OS=Homo sapiens OX=9606 GN=CDK20 PE=1 SV=1
MDQYCILGRIGEGAHGIVFKAKHVETGEIVALKKVALRRLEDGFPNQALREIKALQEMED
NQYVVQLKAVFPHGGGFVLAFEFMLSDLAEVVRHAQRPLAQAQVKSYLQMLLKGVAFCHA
NNIVHRDLKPANLLISASGQLKIADFGLARVFSPDGSRLYTHQVATRWYRAPELLYGARQ
YDQGVDLWSVGCIMGELLNGSPLFPGKNDIEQLCYVLRILGTPNPQVWPELTELPDYNKI
SFKEQVPMPLEEVLPDVSPQALDLLGQFLLYPPHQRIAASKALLHQYFFTAPLPAHPSEL
PIPQRLGGPAPKAHPGPPHIHDFHVDRPLEESLLNPELIRPFILEG

sequence length: 346


## 1. UniProt/BLASTでCDK20に近い蛋白を探す

swissprot(UniProtKB/Swiss-Prot、審査済みエントリ)データベースに対し、Homo sapiensに限定してBLASTサーチを行い、
CDK20_HUMANに配列が近い蛋白(UniProt accession)を集める。


In [5]:
import pickle
from blastsearch import fetch_hits, submit_blast, wait_for_blast

HITS_CACHE = OUTDIR / "blast_hits.pkl"
RID_CACHE = OUTDIR / "blast_rid.txt"

if HITS_CACHE.exists():
    print(f"Using cached BLAST hits: {HITS_CACHE}")
    with open(HITS_CACHE, "rb") as f:
        hits = pickle.load(f)
else:
    if RID_CACHE.exists():
        rid = RID_CACHE.read_text().strip()
        print(f"Resuming existing BLAST job {rid} (submitted previously) ...")
    else:
        rid = submit_blast(sequence, program="blastp", database="swissprot", entrez_query="Homo sapiens[Organism]")
        RID_CACHE.write_text(rid)

    try:
        wait_for_blast(rid, poll_interval=10.0, timeout=600.0)
    except RuntimeError:
        # ジョブ自体が失敗した場合は再投函が必要なのでRIDキャッシュを破棄する
        RID_CACHE.unlink(missing_ok=True)
        raise
    # TimeoutErrorはRID_CACHEを残したまま伝播させる(セルを再実行すれば同じRIDで待機を再開できる)

    hits = fetch_hits(rid)
    with open(HITS_CACHE, "wb") as f:
        pickle.dump(hits, f)
    RID_CACHE.unlink(missing_ok=True)
    print(f"Saved BLAST hits to {HITS_CACHE}")

Using cached BLAST hits: data/cdk20_investigation/blast_hits.pkl


In [6]:
import pandas as pd
from blastsearch import parse_uniprot_subject_id
from uniprot.entry import fetch_entry_names

# UniProt accessionごとに最良ヒット(evalue最小)だけを残す。CDK20_HUMAN自身は除く。
best_by_accession = {}
for h in hits:
    acc = parse_uniprot_subject_id(h["subject_id"])
    if acc == accession:
        continue
    if acc not in best_by_accession or h["evalue"] < best_by_accession[acc]["evalue"]:
        best_by_accession[acc] = {**h, "accession": acc}
unique_hits = sorted(best_by_accession.values(), key=lambda h: h["evalue"])

hits_df = pd.DataFrame(unique_hits)
entry_names = fetch_entry_names(hits_df["accession"].tolist())
hits_df["entry_name"] = hits_df["accession"].map(entry_names)
hits_df["coverage"] = (hits_df["align_length"] / len(sequence) * 100).round(1)
hits_df = hits_df[["accession", "entry_name", "identity", "coverage", "align_length", "evalue", "bit_score"]]
hits_df.to_csv(OUTDIR / "blast_hits.csv", index=False)
print(f"unique candidate proteins: {len(hits_df)}")


def _format_evalue(e: float) -> str:
    """0はそのまま「0」、それ以外は有効数字2桁の指数表記(例: 9.47e-95 -> 9.5e-95)にする。"""
    return "0" if e == 0 else f"{e:.1e}"


display_df = hits_df.head(20).copy()
display_df["identity"] = display_df["identity"].round(2)
display_df["bit_score"] = display_df["bit_score"].round().astype(int)
display_df["evalue"] = display_df["evalue"].apply(_format_evalue)
display_df


02:34:26 [uniprot.entry] Fetching UniProt entry names for 99 accessions ...
02:34:27 [uniprot.entry]   -> 99 entry names resolved


unique candidate proteins: 99


,accession,entry_name,identity,coverage,align_length,evalue,bit_score
0,Q00526,CDK3_HUMAN,45.28,88.7,307,6.8e-86,261
1,P06493,CDK1_HUMAN,43.10,83.8,290,5.0e-80,246
2,P50613,CDK7_HUMAN,43.14,88.4,306,6.8e-80,247
3,P24941,CDK2_HUMAN,43.77,85.8,297,1.1e-79,245
4,Q00535,CDK5_HUMAN,46.02,83.5,289,5.6e-79,243
5,P21127,CD11B_HUMAN,42.16,88.4,306,3.2e-73,241
6,Q9UQ88,CD11A_HUMAN,41.83,88.4,306,7.3e-73,240
7,Q14004,CDK13_HUMAN,38.66,90.5,313,1.0e-69,236
8,P50750,CDK9_HUMAN,40.00,89.6,310,1.2e-68,219
9,Q9NYV4,CDK12_HUMAN,39.10,90.2,312,8.3e-66,224


## 2. 見つかった蛋白ごとにPDBエントリ数・ChEMBL化合物数を集計してランキングする

上位ヒットについて、UniProtのPDB相互参照からPDBエントリ数を、ChEMBLから既知活性化合物数(ユニークな化合物数)を
集計する。両方が揃っている蛋白ほど、この研究(ドッキングテンプレート+SAR参照)で活用しやすいと考えられる。


In [ ]:
import requests

from idmap import resolve_chembl_target_id
from uniprot.entry import fetch_protein_info
from chembl import fetch_activities

TOP_N = 40
candidates_df = hits_df.head(TOP_N).copy()

extra_rows = []
for _, row in candidates_df.iterrows():
    acc = row["accession"]
    try:
        info = fetch_protein_info(acc)
        pdb_count = len(info["pdb_structures"])
    except requests.exceptions.RequestException as e:
        print(f"  [skip] {acc}: failed to fetch UniProt info ({e})")
        continue

    # compound_count: None=取得失敗(ChEMBL障害等で不明)、0=ChEMBL target自体が存在しない(確認済み)
    compound_count = None
    chembl_target_id = None
    try:
        chembl_target_id = resolve_chembl_target_id(acc)
        compounds = fetch_activities(chembl_target_id)
        compound_count = len({a["molecule_chembl_id"] for a in compounds})
    except ValueError:
        compound_count = 0
    except requests.exceptions.RequestException as e:
        print(f"  [warn] {acc}: ChEMBL lookup failed, marking as unknown ({e})")

    extra_rows.append({
        "accession": acc,
        "pdb_count": pdb_count,
        "chembl_target_id": chembl_target_id,
        "compound_count": compound_count,
    })

extra_df = pd.DataFrame(extra_rows)
ranking_df = candidates_df.merge(extra_df, on="accession").sort_values(
    ["pdb_count", "compound_count"], ascending=False
).reset_index(drop=True)
ranking_df.to_csv(OUTDIR / "protein_ranking.csv", index=False)

ranking_display_df = ranking_df.copy()
ranking_display_df["identity"] = ranking_display_df["identity"].round(2)
ranking_display_df["bit_score"] = ranking_display_df["bit_score"].round().astype(int)
ranking_display_df["evalue"] = ranking_display_df["evalue"].apply(_format_evalue)
ranking_display_df


02:34:27 [uniprot.entry] Fetching UniProt entry for Q00526 ...
02:34:28 [idmap.identifiers] Resolving UniProt accession Q00526 -> ChEMBL target id ...
02:34:35 [uniprot.entry] Fetching UniProt entry for P06493 ...


  [warn] Q00526: ChEMBL lookup failed, marking as unknown (500 Server Error: Internal Server Error for url: https://www.ebi.ac.uk/chembl/api/data/target.json?target_components__accession=Q00526&format=json)


02:34:37 [idmap.identifiers] Resolving UniProt accession P06493 -> ChEMBL target id ...
02:35:08 [uniprot.entry] Fetching UniProt entry for P50613 ...


  [warn] P06493: ChEMBL lookup failed, marking as unknown (HTTPSConnectionPool(host='www.ebi.ac.uk', port=443): Read timed out. (read timeout=30))


02:35:09 [idmap.identifiers] Resolving UniProt accession P50613 -> ChEMBL target id ...
02:35:15 [uniprot.entry] Fetching UniProt entry for P24941 ...


...

02:37:08 [idmap.identifiers] Resolving UniProt accession P50750 -> ChEMBL target id ...
02:37:14 [uniprot.entry] Fetching UniProt entry for Q9NYV4 ...


  [warn] P50750: ChEMBL lookup failed, marking as unknown (500 Server Error: Internal Server Error for url: https://www.ebi.ac.uk/chembl/api/data/target.json?target_components__accession=P50750&format=json)


02:37:16 [idmap.identifiers] Resolving UniProt accession Q9NYV4 -> ChEMBL target id ...
02:37:21 [uniprot.entry] Fetching UniProt entry for Q15131 ...


  [warn] Q9NYV4: ChEMBL lookup failed, marking as unknown (500 Server Error: Internal Server Error for url: https://www.ebi.ac.uk/chembl/api/data/target.json?target_components__accession=Q9NYV4&format=json)


02:37:22 [idmap.identifiers] Resolving UniProt accession Q15131 -> ChEMBL target id ...


## 3. CDK20自体の既知活性化合物を収集(ChEMBL)

構造とは独立に、CDK20_HUMANに対する既知の活性化合物(pChEMBL値あり)をChEMBLから取得する。


In [ ]:
chembl_target_id = resolve_chembl_target_id(accession)
activities = fetch_activities(chembl_target_id)

compounds_df = pd.DataFrame(activities)
if not compounds_df.empty:
    compounds_df = compounds_df[
        ["molecule_chembl_id", "molecule_pref_name", "canonical_smiles",
         "standard_type", "standard_value", "standard_units", "pchembl_value",
         "assay_chembl_id", "document_chembl_id"]
    ].sort_values("pchembl_value", ascending=False).reset_index(drop=True)
compounds_df.to_csv(OUTDIR / "known_compounds.csv", index=False)
compounds_df


In [ ]:
print(f"BLAST candidate proteins (unique UniProt accessions): {len(hits_df)}")
print(f"Ranked candidates (top {TOP_N} checked): {len(ranking_df)}")
print(f"  with PDB structures: {(ranking_df['pdb_count'] > 0).sum()}")
print(f"  with ChEMBL compounds: {(ranking_df['compound_count'] > 0).sum()}")
print(f"  with both (usable as docking+SAR reference): {((ranking_df['pdb_count'] > 0) & (ranking_df['compound_count'] > 0)).sum()}")
print(f"Known active compounds for CDK20 itself (ChEMBL, pChEMBL available): {len(compounds_df)}")
